In [1]:
import datetime as dt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import APSIMGraphHelpers as AGH
import GraphHelpers as GH
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.dates as mdates
import MathsUtilities as MUte
import shlex # package to construct the git command to subprocess format
import subprocess 
import xmltodict, json
import sqlite3
from scipy.optimize import basinhopping
from scipy.optimize import dual_annealing
from skopt import gp_minimize
%matplotlib inline

In [2]:
con = sqlite3.connect(r'C:\GitHubRepos\ApsimX\Tests\Validation\Wheat\Wheat.db')
DailyReport = pd.read_sql("Select * from DailyReport",con)

In [3]:
Simulations = pd.read_sql("Select * from _Simulations",con)
Simulations.set_index('ID',inplace=True)

In [43]:
Observed = pd.read_sql("Select * from Observed",con).dropna(axis=1,how='all')
Observed.loc[:,'SimulationName'] = [Simulations.loc[x,'Name'] for x in Observed.SimulationID]
Observed.set_index(['SimulationName','Clock.Today'],drop=False,inplace=True)
Observed.sort_index(inplace=True)
Observed.sort_index(inplace=True,axis=1)

In [44]:
HarvestPred = pd.read_sql("Select * from HarvestReport",con).dropna(axis=1,how='all')
HarvestPred.loc[:,'SimulationName'] = [Simulations.loc[x,'Name'] for x in HarvestPred.SimulationID]
HarvestPred.set_index(['SimulationName','Clock.Today'],drop=False,inplace=True)
HarvestPred.sort_index(inplace=True)
HarvestPred.sort_index(inplace=True,axis=1)

In [37]:
Factors = pd.read_sql("Select * from _Factors",
                    con)
Factors.set_index('SimulationID',drop=False,inplace=True)
Factors = Factors.sort_values(by=['FactorName']).drop_duplicates()
Factors.sort_index(inplace=True)
Factors.sort_index(inplace=True,axis=1)

In [6]:
Simulations

,Name,FolderName
ID,,
1,APS26NRate160WaterWet,SE Queensland
2,APS26NRate40WaterDry,SE Queensland
3,APS26NRate0WaterWet,SE Queensland
4,APS26NRate160WaterDry,SE Queensland
5,APS26NRate40WaterWet,SE Queensland
...,...,...
7960,FAR WAE W22-02MgmtGrazedCvAccroc,
7961,FAR WAE W22-02MgmtHigh InputCvRockstar,
7962,FAR WAE W22-02MgmtHigh InputCvMowhawk,


In [7]:
Factors

,CheckpointID,ExperimentName,FactorName,FactorValue,FolderName,SimulationID
SimulationID,,,,,,
1,1,APS26,Water,Wet,SE Queensland,1
1,1,APS26,NRate,160,SE Queensland,1
2,1,APS26,Water,Dry,SE Queensland,2
2,1,APS26,NRate,40,SE Queensland,2
3,1,APS26,Water,Wet,SE Queensland,3
...,...,...,...,...,...,...
6796,1,PotentialGrainSize,Value,Hartog40,ProteinAccumulation,6796
6797,1,PotentialGrainSize,Value,Hartog44,ProteinAccumulation,6797
6799,1,TerminalWaterStress,Irrigation,Full,TerminalWaterStress,6799


In [38]:
list(Observed.loc[Observed.loc[:,'Wheat.Phenology.CurrentStageName']=='HarvestRipe',:].dropna(axis=1,how='all').columns)

['([Wheat].Leaf.Transpiration + [Soil].SoilWater.Es + [MicroClimate].PrecipitationInterception)',
 'CheckpointID',
 'Clock.Today',
 'NDVIModel.Script.NDVI',
 'NDVIModel.Script.NDVI.se',
 'SimulationID',
 'SimulationName',
 'Soil.Water.Volumetric(1)',
 'Soil.Water.Volumetric(2)',
 'Soil.Water.Volumetric(3)',
 'Soil.Water.Volumetric(4)',
 'Soil.Water.Volumetric(5)',
 'Soil.Water.Volumetric(6)',
 'Soil.Water.Volumetric(7)',
 'Soil.Water.Volumetric(8)',
 'Wheat.AboveGround.N',
 'Wheat.AboveGround.NError',
 'Wheat.AboveGround.Nconc',
 'Wheat.AboveGround.Nconc.se',
 'Wheat.AboveGround.Wt',
 'Wheat.AboveGround.Wt.se',
 'Wheat.AboveGround.WtError',
 'Wheat.DaysAfterSowing',
 'Wheat.Ear.N',
 'Wheat.Ear.Nconc',
 'Wheat.Ear.Wt',
 'Wheat.Grain.Density',
 'Wheat.Grain.Density.se',
 'Wheat.Grain.FWt',
 'Wheat.Grain.FWt15',
 'Wheat.Grain.Moisture',
 'Wheat.Grain.Moisture.se',
 'Wheat.Grain.N',
 'Wheat.Grain.NConc',
 'Wheat.Grain.NError',
 'Wheat.Grain.Number',
 'Wheat.Grain.NumberError',
 'Wheat.Grai

In [47]:
HarvestObs = Observed.loc[Observed.loc[:,'Wheat.Phenology.CurrentStageName'] == 'HarvestRipe',:].copy()
HarvestObs.dropna(axis=1,how='all',inplace=True)
HarvestObs.dropna(axis=0,how='all',inplace=True)

In [48]:
obsData = HarvestObs.loc[:,'Wheat.Grain.Wt'].dropna()

In [53]:
obsData.index = obsData.index.get_level_values(0)

In [54]:
obsData

SimulationName
APS14StubbleBareNRate000       125.983
APS14StubbleBareNRate040       280.967
APS14StubbleBareNRate080       379.033
APS14StubbleBareNRate200       516.483
APS14StubbleLucerneNRate000    352.917
                                ...   
Wongan83SoilUnRippedN50        251.000
YarrabahCreek                  720.800
Yucheng02                      526.060
Yucheng03                      523.530
Yucheng04                      516.930
Name: Wheat.Grain.Wt, Length: 928, dtype: float64

In [58]:
HarvestPred#.reindex(obsData.index).loc[:,'Wheat.Grain.Wt']

,,CO2,Canopy,CheckpointID,Clock.StartDate,Clock.Today,Cm,Cultivar,Cv,Date,Durat,...,Wheat.Phenology.FinalLeafNumber,Wheat.Phenology.FlagLeafDAS,Wheat.Phenology.FloweringDAS,Wheat.Phenology.HeadingDAS,Wheat.Phenology.MaturityDAS,Wheat.Phenology.TerminalSpikeletDAS,Wheat.SowingData.Cultivar,Wheat.SowingDate,Wheat.SowingDate.DayOfYear,Zone
SimulationName,Clock.Today,,,,,,,,,,,,,,,,,,,,,
12JuneeJames-1-100Cultivareaglehawk,2012-11-26 12:00:00,None,None,1,2012-04-17 12:00:00,2012-11-26 12:00:00,None,eaglehawk,None,None,None,...,7.678255,116.0,181.0,174.0,222.0,51.0,eaglehawk,2012-04-18 12:00:00,109,Field
12JuneeJames-1-100Cultivargregory,2012-11-11 12:00:00,None,None,1,2012-04-17 12:00:00,2012-11-11 12:00:00,None,gregory,None,None,None,...,10.020095,133.0,158.0,152.0,207.0,67.0,gregory,2012-04-18 12:00:00,109,Field
12JuneeJames-1-50Cultivareaglehawk,2012-11-26 12:00:00,None,None,1,2012-04-17 12:00:00,2012-11-26 12:00:00,None,eaglehawk,None,None,None,...,7.678255,116.0,181.0,174.0,222.0,51.0,eaglehawk,2012-04-18 12:00:00,109,Field
12JuneeJames-2-100Cultivareaglehawk,2012-11-26 12:00:00,None,None,1,2012-04-25 12:00:00,2012-11-26 12:00:00,None,eaglehawk,None,None,None,...,7.080433,107.0,172.0,165.0,214.0,47.0,eaglehawk,2012-04-26 12:00:00,117,Field
12JuneeJames-2-100Cultivargregory,2012-11-12 12:00:00,None,None,1,2012-04-25 12:00:00,2012-11-12 12:00:00,None,gregory,None,None,None,...,9.543571,128.0,152.0,146.0,200.0,63.0,gregory,2012-04-26 12:00:00,117,Field
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YanYeanTOS9CvYoung,2019-12-21 12:00:00,None,None,1,2019-07-16 12:00:00,2019-12-21 12:00:00,None,None,Young,None,None,...,8.258832,88.0,110.0,105.0,158.0,56.0,Young,2019-07-16 12:00:00,197,Field
YarrabahCreek,2001-12-05 12:00:00,None,None,1,2001-06-19 12:00:00,2001-12-05 12:00:00,None,None,None,None,None,...,10.048463,103.0,117.0,111.0,169.0,67.0,Mercury,2001-06-19 12:00:00,170,Field
Yucheng02,2003-06-15 12:00:00,None,None,1,2002-10-01 12:00:00,2003-06-15 12:00:00,None,None,None,None,None,...,8.420086,189.0,214.0,210.0,243.0,153.0,Keyu13,2002-10-15 12:00:00,288,Field


In [ ]:
HarvestObs.loc[

In [45]:
for i in Observed.index:
    simulationID = Observed.loc[i,'SimulationID']
    Observed.loc[i,['ExperimentName', 'FactorName', 'FactorValue','FolderName']] = Factors.loc[simulationID,['ExperimentName', 'FactorName', 'FactorValue','FolderName']]

ValueError: Incompatible indexer with Series